# get_cctv_url.py
**電腦視覺與深度學習 › Day20 YOLO11n 物件偵測(預訓練到自己訓練)**

這支小工具從台中市 CCTV 的觀看網頁裡,自動挖出真正的串流網址,給 09 和串流測試使用。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/15)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY10"
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), 'get_cctv_url.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

### 原理
觀看網頁裡有一個 img 標籤,它的 src 就是真正的影像串流網址。我們抓下網頁、用正規表示式把它找出來。

In [ ]:
# 從台中 CCTV 觀看頁自動抓實際 MJPEG 串流 URL

### 匯入
re 是正規表示式,sys 讀命令列參數,urllib 抓網頁。

In [ ]:
#
# 用法：
#   python get_cctv_url.py C000002
#
# 原理：
#   台中 CCTV 觀看頁 (Showcctv?id=xxx) 底下有一個 <img src="...">
#   那個 img src 就是實際 MJPEG 串流網址（永久 endpoint，不會過期）
#   我們用 urllib GET 那頁的 HTML，用 regex 挖出 img src

import re
import sys
import urllib.request

### 下載觀看網頁
把攝影機編號組進觀看網址,用 urlopen 下載,最多等 15 秒,再把內容解碼成文字。

### 用正規表示式找網址
這個樣式在找「img 標籤裡、src 是 tcnvr 開頭的網址」。找不到就報錯,可能是編號錯了,或是市府網站改版了。

In [ ]:
def 取得CCTV串流URL(device_id: str) -> str:
    viewer_url = f"https://motoretag.taichung.gov.tw/ATIS_TCC/Device/Showcctv?id={device_id}"
    r = urllib.request.urlopen(viewer_url, timeout=15)
    html = r.read().decode("utf-8", errors="replace")

    # 找 <img src="https://tcnvr..."> 或類似的
    m = re.search(r'<img[^>]+src=["\'](https?://tcnvr[^"\']+)["\']', html)
    if not m:
        raise RuntimeError(
            f"CCTV 頁面 {viewer_url} 裡沒找到 stream img。可能：\n"
            "  1) device_id 錯誤（試 C000001、C000003 等）\n"
            "  2) 台中市府網站結構改了"
        )
    return m.group(1)

### 直接執行
直接執行時可以在後面接攝影機編號,沒給就用 C000002,印出找到的網址。

In [ ]:
if __name__ == "__main__":
    device_id = sys.argv[1] if len(sys.argv) > 1 else "C000002"
    url = 取得CCTV串流URL(device_id)
    print(url)

## 小結
把找網址的工作獨立成一個小工具,其他程式只要匯入它就能用。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/7/15) 看逐句拆解、做練習。